# Too complicated to Model

This notebook is part of my **PyCon ZA 2026** and **Swiss Python Summit 2026** talks, *Missing Female Data*.

Here I prepare a small dataset from my own Garmin training history for the modelling section of the talk.

The data contain information about my training sessions, including activity type, date, pace, heart rate, and position within my **28-day oral-contraceptive schedule**.

I use the variable `cycle_position` to describe where each training session falls within that schedule. It should not be interpreted as a verified biological menstrual-cycle phase or as equivalent to a natural menstrual cycle.

The purpose of this notebook is simply to clean and prepare the data so that the modelling notebook can work with a consistent dataset.

The modelling itself happens separately, where I will compare three ways of representing cycle timing:

1. **Phase categories**
2. **Cyclical terms**
3. **A cyclic GAM**

This is a small, personal dataset. It is not intended to tell us how menstrual cycles affect athletic performance in women generally.

Instead, it gives me a concrete dataset with which to ask a modelling question:

> **What changes when we change the way cycle timing is represented in the model?**


## Data preparation

Before modelling anything, I need to turn the Garmin export into a dataset that I can actually work with.

The preparation steps are:

1. **Load the Garmin export**
   - Read the CSV into pandas.
   - Inspect the columns and data types.

2. **Clean the activity data**
   - Remove records that are not suitable for this analysis.
   - Keep the activity type I want to model.
   - Check for obvious data-entry or recording errors.

3. **Prepare dates and times**
   - Convert the activity timestamp into a proper datetime.
   - Keep the date and time available for later analysis.

4. **Prepare numerical variables**
   - Convert values such as distance, pace and other training metrics into usable numeric formats.
   - Handle Garmin's missing-value markers such as `--`.

5. **Create `cycle_position`**
   - Keep the recorded position within the 28-day oral-contraceptive schedule.
   - Treat this as a schedule position, not as a verified biological menstrual-cycle phase.

6. **Explore the data**
   - Check how many activities I have.
   - Look at the distribution of activity types.
   - Check which cycle positions are represented.
   - Inspect the variables that might be useful for modelling.

7. **Export the modelling dataset**
   - Save the cleaned running data as a separate CSV.
   - Use this cleaned dataset in the modelling notebook.

The goal here is not to answer a research question yet.

It is simply to make the data **consistent, transparent and ready to model**.

#### Import the Libraries

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

#### Read and prepare the data

In [2]:
df=pd.read_csv("activities.csv")
df.head()

,activity_type,date,title,cycle_position,distance,calories,time,avg_HR,max_HR,aerobic _TE,...,body_battery_drain,min_temp,decompression,best_lap_time,number_of_laps,max_temp,moving_time,elapsed_time,min_elevation,max_elevation
0,Running,2026-09-26 10:01:38,Berlin - Breakfast Run,12,4.52,306,0:32:33,175,191,4.3,...,-13,--,No,0:03:40,5,--,0:29:12,0:32:33,33,40
1,HIIT,2026-09-24 20:42:22,HIIT,10,0.00,266,0:33:10,154,187,3.5,...,-6,--,No,0:33:10,1,--,0:33:10,0:33:10,--,--
2,Tennis,2026-09-23 8:01:20,Berlin Tennis,9,2.51,326,0:59:51,130,182,2.8,...,-60,--,No,0:59:51,1,--,0:39:10,0:59:51,--,--
3,Running,2026-09-20 11:01:24,Berlin - Rehberge Viertelmarathon,6,10.24,644,1:14:25,176,202,5.0,...,-21,--,No,0:00:45,12,--,1:14:23,1:14:25,31,52
4,HIIT,2026-09-19 9:39:20,HIIT,5,0.00,296,0:44:18,143,187,3.2,...,-10,--,No,0:44:18,1,--,0:44:18,0:44:18,--,--


In [3]:
#seperating date and time
df["datetime"] = pd.to_datetime(df["date"])
df["date"] = df["datetime"].dt.date
df["time"] = df["datetime"].dt.time

In [4]:
#double check
df[["datetime","date","time"]].head()

,datetime,date,time
0,2026-09-26 10:01:38,2026-09-26,10:01:38
1,2026-09-24 20:42:22,2026-09-24,20:42:22
2,2026-09-23 08:01:20,2026-09-23,08:01:20
3,2026-09-20 11:01:24,2026-09-20,11:01:24
4,2026-09-19 09:39:20,2026-09-19,09:39:20


In [5]:
df["activity_type"].value_counts()

activity_type
Running                25
HIIT                   21
Strength Training       9
Open Water Swimming     4
Tennis                  3
Indoor Cycling          1
Name: count, dtype: int64

In [7]:
df["cycle_position"].value_counts().sort_index()

cycle_position
0     2
1     3
2     3
3     2
5     3
6     5
7     3
8     1
9     3
10    3
11    3
12    1
13    3
14    3
15    1
16    1
17    2
18    3
19    1
21    3
22    2
23    2
24    3
25    1
26    1
27    3
28    2
Name: count, dtype: int64

No record on day 4 or 20.

In [8]:
pd.crosstab(df["cycle_position"], df["activity_type"])

activity_type,HIIT,Indoor Cycling,Open Water Swimming,Running,Strength Training,Tennis
cycle_position,,,,,,
0,2,0,0,0,0,0
1,1,0,0,2,0,0
2,1,0,0,2,0,0
3,0,0,0,1,0,1
5,1,0,1,0,1,0
6,1,0,1,2,1,0
7,2,1,0,0,0,0
8,0,0,0,1,0,0
9,1,0,0,1,0,1


In [10]:
df.groupby("activity_type").count().T #transpose

activity_type,HIIT,Indoor Cycling,Open Water Swimming,Running,Strength Training,Tennis
date,21,1,4,25,9,3
title,21,1,4,25,9,3
cycle_position,21,1,4,25,9,3
distance,21,1,4,25,9,3
calories,21,1,4,25,9,3
time,21,1,4,25,9,3
avg_HR,21,1,4,25,9,3
max_HR,21,1,4,25,9,3
aerobic _TE,21,1,4,25,9,3
avg_run_cadence,21,1,4,25,9,3


In [11]:
df = df.replace("--", pd.NA)

In [12]:
# I want to see which columns HIIT and running have in common so I can decise what to use for my analyse
df[df["activity_type"].isin(["Running", "HIIT"])].groupby("activity_type").count().T

activity_type,HIIT,Running
date,21,25
title,21,25
cycle_position,21,25
distance,21,25
calories,21,25
time,21,25
avg_HR,21,25
max_HR,21,25
aerobic _TE,21,25
avg_run_cadence,0,25


In [13]:
# we are only going to focus on running
running = df[df["activity_type"] == "Running"].copy()

In [14]:
running.to_csv("running.csv", index=False)